In [ ]:

import argparse
import colorsys
import json
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
from PIL import Image
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


# --------------------------------------------------------------------------
# Color utilities
# --------------------------------------------------------------------------
def rgb_to_hex(rgb):
    r, g, b = (int(round(c)) for c in rgb)
    return f"#{r:02x}{g:02x}{b:02x}"


def rgb_to_hls(rgb):
    """RGB (0-255) -> (h in degrees 0-360, l 0-1, s 0-1)."""
    r, g, b = (c / 255.0 for c in rgb)
    h, l, s = colorsys.rgb_to_hls(r, g, b)
    return h * 360.0, l, s


def hls_to_rgb(h, l, s):
    """(h degrees, l 0-1, s 0-1) -> RGB (0-255)."""
    r, g, b = colorsys.hls_to_rgb((h % 360) / 360.0, min(max(l, 0), 1), min(max(s, 0), 1))
    return (r * 255, g * 255, b * 255)


def is_neutral(rgb, sat_thresh=0.15, light_low=0.12, light_high=0.92):
    """Black, white, and gray (low saturation or extreme lightness) are neutrals."""
    _, l, s = rgb_to_hls(rgb)
    return s < sat_thresh or l < light_low or l > light_high


HUE_NAMES = [
    (15, "red"), (45, "orange"), (70, "yellow"), (160, "green"),
    (200, "teal"), (255, "blue"), (290, "purple"), (335, "pink"), (361, "red"),
]


def name_color(rgb):
    """Rough human-readable color name from HLS values."""
    h, l, s = rgb_to_hls(rgb)
    if l < 0.12:
        return "black"
    if l > 0.92:
        return "white"
    if s < 0.15:
        return "dark gray" if l < 0.35 else "light gray" if l > 0.7 else "gray"
    if 15 <= h < 45 and l < 0.40:
        base = "brown"
    elif 15 <= h < 45 and s < 0.45 and l > 0.6:
        base = "beige"
    else:
        base = next(n for limit, n in HUE_NAMES if h < limit)
    if l < 0.30:
        return f"dark {base}"
    if l > 0.72:
        return f"light {base}"
    return base


# --------------------------------------------------------------------------
# Step 1: Dominant color extraction with K-Means
# --------------------------------------------------------------------------
def load_pixels(image_path, max_side=150, remove_background=True):
    """Load image, downsize for speed, drop transparent / plain-background pixels."""
    img = Image.open(image_path).convert("RGBA")
    img.thumbnail((max_side, max_side))
    arr = np.array(img)
    rgb, alpha = arr[..., :3].astype(float), arr[..., 3]

    mask = alpha > 128  # ignore transparent pixels (PNG cut-outs)

    if remove_background:
        # If the image border is a near-uniform color (studio backdrop), remove it.
        border = np.concatenate([rgb[0], rgb[-1], rgb[:, 0], rgb[:, -1]])
        if border.std(axis=0).mean() < 18:
            bg = np.median(border, axis=0)
            dist = np.linalg.norm(rgb - bg, axis=2)
            fg = mask & (dist > 40)
            if fg.sum() > 0.05 * mask.sum():  # don't strip the whole garment
                mask = fg

    pixels = rgb[mask]
    if len(pixels) == 0:
        pixels = rgb.reshape(-1, 3)
    return pixels


def choose_k(pixels, k_min=2, k_max=6, sample=2000, seed=42):
    """Pick the number of clusters using the silhouette score."""
    rng = np.random.default_rng(seed)
    if len(pixels) > sample:
        pixels = pixels[rng.choice(len(pixels), sample, replace=False)]
    if len(np.unique(pixels, axis=0)) < k_min + 1:
        return 1
    best_k, best_score = k_min, -1
    for k in range(k_min, min(k_max, len(np.unique(pixels, axis=0)) - 1) + 1):
        labels = KMeans(n_clusters=k, n_init=5, random_state=seed).fit_predict(pixels)
        if len(set(labels)) < 2:
            continue
        score = silhouette_score(pixels, labels)
        if score > best_score:
            best_k, best_score = k, score
    return best_k


@dataclass
class DominantColor:
    rgb: tuple
    hex: str
    percent: float
    name: str
    neutral: bool


def extract_dominant_colors(image_path, k=None, max_k=6, min_percent=3.0):
    """Return dominant colors sorted by share of the garment (largest first)."""
    pixels = load_pixels(image_path)
    k = k or choose_k(pixels, k_max=max_k)
    k = max(1, min(k, len(np.unique(pixels, axis=0))))

    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(pixels)
    counts = np.bincount(km.labels_, minlength=k)
    shares = counts / counts.sum() * 100

    colors = []
    for center, pct in sorted(zip(km.cluster_centers_, shares), key=lambda x: -x[1]):
        if pct < min_percent:
            continue
        rgb = tuple(int(round(c)) for c in center)
        colors.append(DominantColor(rgb, rgb_to_hex(rgb), round(float(pct), 1),
                                    name_color(rgb), is_neutral(rgb)))
    return colors


# --------------------------------------------------------------------------
# Step 2: Color theory recommendations
# --------------------------------------------------------------------------
def rotate_hue(rgb, degrees, l_shift=0.0, s_scale=1.0):
    h, l, s = rgb_to_hls(rgb)
    return hls_to_rgb(h + degrees, l + l_shift, s * s_scale)


def build_schemes(rgb):
    """Classic color-wheel harmonies based on a single base color."""
    h, l, s = rgb_to_hls(rgb)
    # Keep suggestions wearable: avoid ultra-saturated / ultra-light results.
    s_adj = min(max(s, 0.30), 0.75)
    l_adj = min(max(l, 0.30), 0.65)

    def make(*offsets):
        return [rgb_to_hex(hls_to_rgb(h + o, l_adj, s_adj)) for o in offsets]

    return {
        "complementary": make(180),
        "analogous": make(-30, 30),
        "triadic": make(120, 240),
        "split_complementary": make(150, 210),
        "tetradic (square)": make(90, 180, 270),
        "monochromatic": [rgb_to_hex(hls_to_rgb(h, ll, s_adj)) for ll in (0.25, 0.45, 0.75)],
    }


NEUTRAL_PAIRINGS = {
    "dark": ["#ffffff", "#f5f5dc", "#c0c0c0"],        # white, beige, light gray
    "light": ["#1f2a44", "#36454f", "#000000"],       # navy, charcoal, black
    "mid": ["#ffffff", "#000000", "#c2b280"],         # white, black, khaki
}

# Classic accent colors that work on any neutral outfit (navy, burgundy, olive, camel...)
NEUTRAL_ACCENTS = ["#1f3a5f", "#7b2d3b", "#556b2f", "#c19a6b", "#2e8b8b"]


def recommend(colors):
    """Build outfit suggestions from the extracted dominant colors."""
    chromatic = [c for c in colors if not c.neutral]
    result = {"garment_colors": [c.__dict__ for c in colors]}

    if chromatic:
        base = chromatic[0]  # most dominant non-neutral color drives the harmony
        schemes = build_schemes(base.rgb)
        result["base_color"] = {"hex": base.hex, "name": base.name}
        result["schemes"] = {
            scheme: [{"hex": h, "name": name_color(tuple(int(h[i:i + 2], 16) for i in (1, 3, 5)))}
                     for h in hexes]
            for scheme, hexes in schemes.items()
        }
        result["neutral_pairings"] = NEUTRAL_PAIRINGS[_light_group(base.rgb)]
        result["tip"] = (
            f"{base.name.title()} is the focal color. Use complementary or split-complementary "
            "colors as accents (≈10%), analogous colors for a calm look, and neutrals for the rest "
            "(60-30-10 rule)."
        )
    else:
        base = colors[0]
        result["base_color"] = {"hex": base.hex, "name": base.name}
        result["schemes"] = {}
        result["neutral_pairings"] = NEUTRAL_PAIRINGS[_light_group(base.rgb)]
        result["accent_suggestions"] = NEUTRAL_ACCENTS
        result["tip"] = (
            f"This item is neutral ({base.name}), so it pairs with almost anything. "
            "Add one accent color to bring the outfit to life."
        )
    return result


def _light_group(rgb):
    _, l, _ = rgb_to_hls(rgb)
    return "dark" if l < 0.35 else "light" if l > 0.65 else "mid"


# --------------------------------------------------------------------------
# Visualization
# --------------------------------------------------------------------------
def plot_result(image_path, colors, rec, save_path=None):
    import matplotlib.pyplot as plt

    n_rows = 2 + len(rec["schemes"]) + (1 if rec.get("accent_suggestions") else 0)
    fig, axes = plt.subplots(n_rows, 1, figsize=(8, 1.2 * n_rows + 3),
                             gridspec_kw={"height_ratios": [4, 1] + [1] * (n_rows - 2)})
    axes[0].imshow(Image.open(image_path).convert("RGB"))
    axes[0].axis("off")
    axes[0].set_title(Path(image_path).name)

    def swatch(ax, hexes, labels, title):
        for i, (h, lab) in enumerate(zip(hexes, labels)):
            ax.add_patch(plt.Rectangle((i, 0), 1, 1, color=h))
            ax.text(i + 0.5, 0.5, lab, ha="center", va="center", fontsize=8,
                    color="white" if rgb_to_hls(tuple(int(h[j:j + 2], 16) for j in (1, 3, 5)))[1] < 0.55 else "black")
        ax.set_xlim(0, max(len(hexes), 1)); ax.set_ylim(0, 1); ax.axis("off")
        ax.set_title(title, loc="left", fontsize=9)

    swatch(axes[1], [c.hex for c in colors], [f"{c.hex}\n{c.percent}%" for c in colors], "Dominant colors")
    row = 2
    for scheme, items in rec["schemes"].items():
        swatch(axes[row], [i["hex"] for i in items], [f"{i['hex']}\n{i['name']}" for i in items], scheme)
        row += 1
    if rec.get("accent_suggestions"):
        acc = rec["accent_suggestions"]
        swatch(axes[row], acc, acc, "Accent suggestions")

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150)
    else:
        plt.show()
    plt.close(fig)


# --------------------------------------------------------------------------
# Dataset handling
# --------------------------------------------------------------------------
DEFAULT_DRIVE_URL = "https://drive.google.com/drive/folders/1a0tNSl8JZwXZ-Zz0CU43hyLpPRA7qSa7?usp=sharing"


def download_drive_folder(url, out_dir="AI_Stylist_Cleaned_Images"):
    """Download a public Google Drive folder with gdown (pip install gdown)."""
    try:
        import gdown
    except ImportError:
        raise SystemExit("gdown is missing. Run: pip install gdown")

    out = Path(out_dir)
    if out.exists() and any(out.rglob("*")):
        print(f"[info] '{out_dir}' already exists, skipping download.")
        return out
    out.mkdir(parents=True, exist_ok=True)
    print(f"[info] Downloading Drive folder to '{out_dir}' ...")
    files = gdown.download_folder(url=url, output=str(out), quiet=False,
                              use_cookies=False)
    if not files:
        raise SystemExit(
            "Download failed. Make sure the folder is shared as 'Anyone with the link', "
            "or download it manually as a ZIP and use --input <folder>."
        )
    return out


def collect_images(input_path, path_col=None):
    p = Path(input_path)
    if p.is_dir():
        return sorted(f for f in p.rglob("*") if f.suffix.lower() in IMAGE_EXTS)
    if p.suffix.lower() == ".csv":
        import pandas as pd
        df = pd.read_csv(p)
        col = path_col or next((c for c in df.columns if "path" in c.lower() or "image" in c.lower()), None)
        if col is None:
            raise ValueError(f"Specify the image-path column with --path-col. Columns: {list(df.columns)}")
        base = p.parent
        return [Path(x) if Path(x).is_absolute() else base / x for x in df[col].dropna()]
    return [p]


def main():
    ap = argparse.ArgumentParser(description="Dominant colors + color-theory outfit suggestions")
    ap.add_argument("--input", help="Image file, folder of images, or CSV with image paths")
    ap.add_argument("--drive-url", nargs="?", const=DEFAULT_DRIVE_URL,
                    help="Public Google Drive folder URL to download first (no value = your AI_Stylist folder)")
    ap.add_argument("--download-dir", default="AI_Stylist_Cleaned_Images", help="Where to save the Drive images")
    ap.add_argument("--limit", type=int, help="Only process the first N images (useful for testing)")
    ap.add_argument("--path-col", help="Column in the CSV that holds image paths")
    ap.add_argument("--k", type=int, help="Fixed number of clusters (default: auto via silhouette)")
    ap.add_argument("--max-k", type=int, default=6)
    ap.add_argument("--out", help="Output .csv or .json file for batch results")
    ap.add_argument("--plot", action="store_true", help="Save a palette PNG for each image")
    ap.add_argument("--plot-dir", default="palettes", help="Folder for palette PNGs")
    args = ap.parse_args()

    if not args.input and not args.drive_url:
        args.drive_url = DEFAULT_DRIVE_URL          # default: your Drive folder
    if not args.out:
        args.out = "results.csv"
    if args.drive_url:
        args.input = str(download_drive_folder(args.drive_url, args.download_dir))

    images = collect_images(args.input, args.path_col)
    if args.limit:
        images = images[:args.limit]
    print(f"[info] Found {len(images)} image(s)")

    rows = []
    for img_path in images:
        try:
            colors = extract_dominant_colors(img_path, k=args.k, max_k=args.max_k)
            rec = recommend(colors)
        except Exception as e:
            print(f"[skip] {img_path}: {e}")
            continue

        print(f"\n{img_path}")
        print("  Dominant:", ", ".join(f"{c.hex} {c.name} ({c.percent}%)" for c in colors))
        for scheme, items in rec["schemes"].items():
            print(f"  {scheme:<20}", ", ".join(f"{i['hex']} {i['name']}" for i in items))
        print("  Neutrals:", ", ".join(rec["neutral_pairings"]))
        if rec.get("accent_suggestions"):
            print("  Accents: ", ", ".join(rec["accent_suggestions"]))
        print("  Tip:", rec["tip"])

        if args.plot:
            Path(args.plot_dir).mkdir(exist_ok=True)
            plot_result(img_path, colors, rec, save_path=str(Path(args.plot_dir) / f"{Path(img_path).stem}_palette.png"))

        rows.append({"image": str(img_path), **rec})

    if args.out and rows:
        if args.out.endswith(".json"):
            Path(args.out).write_text(json.dumps(rows, indent=2))
        else:
            import pandas as pd
            pd.DataFrame([{
                "image": r["image"],
                "dominant_hex": ";".join(c["hex"] for c in r["garment_colors"]),
                "dominant_names": ";".join(c["name"] for c in r["garment_colors"]),
                "dominant_percent": ";".join(str(c["percent"]) for c in r["garment_colors"]),
                "base_color": r["base_color"]["name"],
                "complementary": ";".join(i["hex"] for i in r["schemes"].get("complementary", [])),
                "analogous": ";".join(i["hex"] for i in r["schemes"].get("analogous", [])),
                "triadic": ";".join(i["hex"] for i in r["schemes"].get("triadic", [])),
                "neutral_pairings": ";".join(r["neutral_pairings"]),
                "tip": r["tip"],
            } for r in rows]).to_csv(args.out, index=False)
        print(f"\nSaved results to {args.out}")


if __name__ == "__main__":
    main()